# Lesson 18: Storing Articles Locally

## Why Do We Need Storage?

In previous lessons, when an agent created an article, the result appeared on screen and **disappeared**. Close the program, everything is lost.

In practice, we need to:
- **Store** articles that have been created
- **Track** metadata (topic, keywords, status, word count)
- **Search** and **retrieve** articles by ID
- **Update** articles after creation (e.g., adding images)

Our product uses **local file storage** — simple, no external services needed.

## Local Files — The Simplest Database

Our storage approach:
- `content/articles.json` — metadata for all articles (topics, keywords, status, word count)
- `content/{id}.md` — the full Markdown content for each article

Comparison:
- **Without storage**: Data lost when program closes
- **With local storage**: Data saved to disk, survives restarts

Why local files?
- **No setup** — no API keys, no cloud accounts, no database servers
- **Simple** — JSON and Markdown are human-readable formats
- **Fast** — no network calls, just disk I/O
- **Portable** — copy the `content/` folder to back up everything

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## Article IDs — Keyword Slugs

Each article has a unique ID based on its keywords or topic.

Instead of abstract IDs like `recABC123` or auto-incrementing integers like `1, 2, 3`, our IDs are **keyword slugs**:

| Topic | Article ID (slug) |
|-------|------------------|
| "On-Page SEO Meta Tags" | `on-page-seo-meta-tags` |
| "How to Write Better Content" | `how-to-write-better-content` |
| "Technical SEO Guide 2026" | `technical-seo-guide-2026` |

The slug is generated from the keywords (or topic if no keywords). This makes article IDs **human-readable** — you can tell what the article is about just from its ID.

If a slug already exists, a number is appended: `on-page-seo-2`, `on-page-seo-3`, etc.

## The Storage Layer

The `tools/storage.py` file provides 4 agent-facing functions:

| Function | Purpose |
|----------|---------|
| `save_article(topic, article_markdown, keywords)` | Save a new article to disk |
| `list_all_articles(status_filter)` | List all articles (metadata only) |
| `get_article_content(article_id)` | Get one article with full content |
| `update_article_content(article_id, article_markdown)` | Update an article's content |

All functions return **JSON strings** — this is how Agno tools work. The agent reads JSON, the function returns JSON.

Let's import and use them:

In [ ]:
from tools.storage import (
    save_article,
    list_all_articles,
    get_article_content,
    update_article_content,
)

print("Storage functions imported!")
print("  save_article(topic, article_markdown, keywords) -> JSON")
print("  list_all_articles(status_filter) -> JSON")
print("  get_article_content(article_id) -> JSON")
print("  update_article_content(article_id, article_markdown) -> JSON")
print()
print(f"Storage directory: content/")
print(f"Metadata file: content/articles.json")
print(f"Article files: content/{{id}}.md")

## Save an Article

`save_article()` takes a topic, Markdown content, and optional keywords. It:
1. Generates a slug ID from the keywords
2. Writes the Markdown to `content/{id}.md`
3. Adds metadata to `content/articles.json`
4. Returns JSON with the article ID and details

In [ ]:
import json

# Save a test article
result = save_article(
    topic="Test Article from Notebook",
    article_markdown="# Test Article\n\nThis is a test article created from the notebook.\n\n## Section 1\n\nSome content here.",
    keywords="test, notebook",
)

article_info = json.loads(result)
print("Saved article:")
print(f"  ID: {article_info['article_id']}")
print(f"  Topic: {article_info['topic']}")
print(f"  File: {article_info['filename']}")
print(f"  Words: {article_info['word_count']}")
print(f"  Status: {article_info['status']}")
print()
print(f"Notice: ID is a slug ('test-notebook'), not a random string or integer.")

## List All Articles

`list_all_articles()` returns metadata for every article (without loading full content — that would be slow for many articles).

In [ ]:
result = list_all_articles()
articles = json.loads(result)

print(f"Total articles: {len(articles)}\n")
for a in articles:
    print(f"  {a['id']}: {a['topic']}")
    print(f"    Status: {a['status']}, Words: {a.get('word_count', 'N/A')}")

## Get Article Content

`get_article_content()` loads the full Markdown from the `.md` file.

In [ ]:
# Get our test article's content
article_id = article_info['article_id']  # From the save above

result = get_article_content(article_id)
content = json.loads(result)

print(f"Article: {content['topic']}")
print(f"\nContent:\n{content['article_markdown']}")

## Update Article Content

`update_article_content()` replaces the article's Markdown and recalculates the word count. This is how the Image Finder adds images — it reads the article, inserts image links, then saves the updated version.

In [ ]:
# Update the test article with more content
updated_markdown = """# Test Article

This is the updated version with more content.

## Section 1: Introduction

This article was created and then updated from a Jupyter notebook.

## Section 2: Key Points

- Point one: storage is simple
- Point two: local files work great
- Point three: no external services needed

## Conclusion

Local file storage is perfect for this kind of project.
"""

result = update_article_content(article_id, updated_markdown)
updated = json.loads(result)
print(f"Updated article {updated['article_id']}")
print(f"New word count: {updated['word_count']}")

## Looking at the Raw Files

Since articles are stored as plain files, you can view them directly:

In [ ]:
# Read the raw articles.json metadata file
metadata_path = os.path.abspath("../../content/articles.json")

if os.path.exists(metadata_path):
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    
    print("content/articles.json:")
    print(json.dumps(metadata, indent=2, ensure_ascii=False)[:2000])
else:
    print("No articles.json file yet.")

## Thread Safety

The storage module uses a `threading.Lock` to prevent data corruption when multiple agents save articles simultaneously (during batch processing).

```python
_lock = threading.Lock()

with _lock:
    # Only one thread can be here at a time
    metadata = _load_metadata()
    metadata[article_id] = {...}
    _save_metadata(metadata)
```

Without the lock, two agents saving at the same time could overwrite each other's data. The lock ensures one finishes before the other starts.

You don't need to worry about this when using the functions — the lock is handled internally.

## SQLite for Chat Memory

Local file storage handles **articles**. But the product also uses **SQLite** for one thing: **chat session memory**.

In `chat.py` and `team.py`:
```python
db=SqliteDb(db_file="chat_sessions.db")
```

This is Agno's built-in storage for conversation history. It's a separate concern from article storage — SQLite handles chat memory, local files handle articles.

## Summary

- **Local file storage** stores articles as `.md` files with JSON metadata
- **Article IDs are keyword slugs** — human-readable, like `on-page-seo-meta-tags`
- **4 key functions**: `save_article()`, `list_all_articles()`, `get_article_content()`, `update_article_content()`
- **All return JSON strings** — the standard format for agent tools
- **Thread-safe** — `threading.Lock` prevents data corruption during batch processing
- **SQLite** is only used for chat memory, not articles

**Next lesson:** How everything connects — tracing a request from the web interface through the team to storage.

## Exercise

Using the storage functions imported above, write code that:

1. Creates a new article with a topic and keywords of your choice
2. Reads the article back and prints its topic and word count
3. Updates the article content with additional text
4. Reads the article again and verifies the word count changed

This is the same pattern the Content Writer agent uses — save, read, update.

In [ ]:
# Exercise: Fill in the blanks to work with local storage

# 1. Create an article (fill in topic and keywords)
result = save_article(___, ___, keywords=___)
my_article = json.loads(result)
my_id = my_article[___]
print(f"Created article: {my_id}")

# 2. Read the article back
result = ___(my_id)
content = json.loads(result)
print(f"Topic: {content[___]}")
print(f"Word count: {len(content['article_markdown'].split())}")

# 3. Update the content
new_content = content['article_markdown'] + "\n\n## New Section\n\nAdded by the exercise."
result = update_article_content(my_id, ___)
updated = json.loads(result)
print(f"\nAfter update:")
print(f"Word count: {updated[___]}")

<details>
<summary>Click to reveal answer</summary>

```python
# 1. Create an article
result = save_article("SEO Tips for Small Businesses", "# SEO Tips\n\nHere are some tips for small businesses.", keywords="seo tips, small business")
my_article = json.loads(result)
my_id = my_article["article_id"]
print(f"Created article: {my_id}")

# 2. Read the article back
result = get_article_content(my_id)
content = json.loads(result)
print(f"Topic: {content['topic']}")
print(f"Word count: {len(content['article_markdown'].split())}")

# 3. Update the content
new_content = content['article_markdown'] + "\n\n## New Section\n\nAdded by the exercise."
result = update_article_content(my_id, new_content)
updated = json.loads(result)
print(f"\nAfter update:")
print(f"Word count: {updated['word_count']}")
```
</details>